# pc2beam — IFC → HELIOS++ simulation demo

> Compatibility notebook path. Canonical location: `tools/helios_generation/notebooks/simulation_demo.ipynb`.

This notebook walks through:

1. Tessellating IFC geometry and exporting **Wavefront OBJ** for HELIOS++ (default: **one OBJ per IFC instance** so LAS `hitObjectId` matches scene parts; optional merged mesh via `helios_one_obj_per_instance=False`).
2. Writing a **HELIOS++** multi-part scene + survey XML driven by `config/scanners_example.yaml`.
3. Optionally running the **`helios`** CLI to produce a **LAS** point cloud, then visualizing the simulated scan **colored by instance id** when `hitObjectId` varies.

**Local (reproducible):** create the Conda env from `environment.yml` (pins **`helios`** and **`ifcopenshell`**). Stock HELIOS data is usually under `$CONDA_PREFIX/share/helios` or bundled as **`pyhelios`** in `site-packages`; `pc2beam` auto-resolves that. Set **`HELIOS_DATA_PATH`** only for a custom HELIOS data root (folder whose `data/` tree includes `platforms.xml`). See [HELIOS++](https://github.com/3dgeo-heidelberg/helios) upstream.

**Google Colab:** pip installs in this notebook do **not** include the `helios` binary — keep **`RUN_HELIOS = False`** unless you use a custom runtime with HELIOS++.

---
**Note:** the notebook detects Google Colab vs local execution, similar to `demo.ipynb`.

## environment detection and setup

In [1]:
try:
    import google.colab
    IN_COLAB = True
    print("environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("environment: local (expected Conda env: pc2beam, Python 3.12)")

if IN_COLAB:
    print("installing dependencies...")
    !pip install -q ifcopenshell plotly omegaconf numpy laspy
    !rm -rf /content/pc2beam
    !git clone https://github.com/fnoi/pc2beam.git
    import sys
    sys.path.insert(0, "/content/pc2beam")
    project_root = __import__("pathlib").Path("/content/pc2beam")
    print("HELIOS++ is not installed on Colab by default; keep `RUN_HELIOS = False` or use a custom runtime.")
else:
    import sys
    from pathlib import Path

    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

ifc_path = project_root / "data" / "model_0_z_up.ifc"
scanners_yaml = project_root / "config" / "scanners_example.yaml"
print("IFC:", ifc_path)
print("scanners:", scanners_yaml)

environment: local (expected Conda env: pc2beam, Python 3.12)
IFC: /Users/fnoi/Code/pc2beam/data/model_0_z_up.ifc
scanners: /Users/fnoi/Code/pc2beam/config/scanners_example.yaml


## imports

In [2]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from omegaconf import OmegaConf

from pc2beam.helios_pipeline import run_ifc_helios_pipeline
from pc2beam.helios_runner import resolve_helios_data_root
from pc2beam.ifc_io import iter_beam_records
from pc2beam.ifc_mesh_export import load_mesh_for_preview
from pc2beam.ifc_mesh_export import instance_id_for_las_hit_object_id
from pc2beam.las_io import find_first_las, las_dimension_names, read_las_scan
from pc2beam.viz import plot_point_cloud, plot_scanners_and_mesh

## beam metadata summary (IFC)

In [3]:
records = iter_beam_records(ifc_path)
print(f"IfcBeam count: {len(records)}")
profiles = {}
for r in records:
    key = (r.profile_name, r.material_name)
    profiles[key] = profiles.get(key, 0) + 1
print("profile / material → beam count (first 15 keys):")
for i, (k, c) in enumerate(sorted(profiles.items(), key=lambda x: -x[1])):
    if i >= 15:
        break
    print(f"  {k[0]!r} + {k[1]!r}: {c}")

IfcBeam count: 80
profile / material → beam count (first 15 keys):
  'IPE 240' + 'S235 | EN 10025-2:2004-11': 30
  'HE 140 A' + 'S235 | EN 10025-2:2004-11': 14
  'HE 260 A' + 'S235 | EN 10025-2:2004-11': 10
  'HE 100 A' + 'S235 | EN 10025-2:2004-11': 10
  'IPE 140' + 'S235 | EN 10025-2:2004-11': 9
  'HE 180 A' + 'S235 | EN 10025-2:2004-11': 4
  'HE 240 A' + 'S235 | EN 10025-2:2004-11': 2
  'HE 200 A' + 'S235 | EN 10025-2:2004-11': 1


## build HELIOS++ asset bundle (OBJ + XML)

Set `RUN_HELIOS = True` when the `helios` executable is on `PATH` and `resolve_helios_data_root()` succeeds (Conda env from `environment.yml`, or `HELIOS_DATA_PATH` for a custom install).

In [4]:
RUN_HELIOS = True

try:
    print("HELIOS data:", resolve_helios_data_root())
    helios_ready = True
except FileNotFoundError as e:
    print(e)
    helios_ready = False

if RUN_HELIOS and not helios_ready:
    RUN_HELIOS = False
    print("Disabling RUN_HELIOS: HELIOS++ data directory not found.")

result = run_ifc_helios_pipeline(
    ifc_path=ifc_path,
    scanners_yaml=scanners_yaml,
    output_root=project_root / "output" / "helios_pc2beam",
    run_simulation=RUN_HELIOS,
    helios_data_path=None,
    helios_one_obj_per_instance=True,
    mesher_linear_deflection=0.02,
)

print("run_dir:", result["run_dir"])
print("survey_xml:", result["survey_xml"])
print("ground_truth_yaml:", result.get("ground_truth_yaml"))
print("helios_one_obj_per_instance:", result.get("helios_one_obj_per_instance"))
print("scene_obj (merged):", result.get("scene_obj"))
print("instances_dir:", result.get("instances_dir"))
print("meshed beams:", result["sidecar"].meshed_beam_count, "/", result["sidecar"].beam_count)
print("pc2beam_input_txt:", result.get("pc2beam_input_txt"))
if result.get("returncode") is not None:
    print("helios return code:", result["returncode"])

HELIOS data: /opt/miniconda3/envs/pc2beam/lib/python3.11/site-packages/pyhelios
HELIOS++ VERSION 2.1.0

CWD: "/Users/fnoi/Code/pc2beam/notebooks"
seed: AUTO
surveyPath: "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/surveys/pc2beam_survey.xml"
assetsPath: ["/opt/miniconda3/envs/pc2beam/lib/python3.11/site-packages/pyhelios", "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a", "/Users/fnoi/Code/pc2beam/notebooks", "/opt/miniconda3/envs/pc2beam/lib/python3.11/site-packages/pyhelios", "/opt/miniconda3/envs/pc2beam/lib/python3.11/site-packages/pyhelios/data", "assets/", ]
outputPath: "/Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/sim_output/"
writeWaveform: 0
writePulse: 0
calcEchowidth: 0
fullWaveNoise: 0
splitByChannel: 0
parallelization: 1
njobs: 0
chunkSize: 32
warehouseFactor: 4
platformNoiseDisabled: 0
legNoiseDisabled: 0
rebuildScene: 0
writeScene: 1
lasOutput: 1
las10: 0
fixedIncidenceAngle: 0
gpsStartTime: 
kdtType: 4
kdtJobs: 0
kdtGeomJobs: 

## output artifacts explained (with examples)

Each simulation run writes four artifact groups under `result["run_dir"]` (plus one file beside the IFC):

1. **Geometry for HELIOS++**
   - In split mode (`helios_one_obj_per_instance=True`): one OBJ per meshed IFC instance in `data/sceneparts/pc2beam/instances/`.
   - In merged mode: one `scene.obj` (or legacy `beams.obj`) plus matching MTL.

2. **Mapping from LAS `hitObjectId` to IFC beam/instance metadata**
   - Available in split mode via sidecar fields such as
     `meshed_instance_ids_in_part_order`, `instance_hit_mapping`, and `beam_instance_hit_mapping`.
   - `hitObjectId` is usually the 0-based HELIOS scene part index.

3. **Point cloud outputs**
   - LAS files in `sim_output/.../legXXX_points.las` when `RUN_HELIOS=True` and HELIOS++ is installed.
   - A demo-compatible TXT in `pc2beam_input/points_with_normals_instances.txt` with columns `x y z nx ny nz instance_id`.

4. **Ground truth YAML**
   - `<ifc_stem>_gt.yaml` beside the IFC input (for example `data/model_0_z_up_gt.yaml`).

The next cell prints concrete examples from this run.

In [5]:
from pathlib import Path

run_dir = Path(result["run_dir"])
sidecar = result["sidecar"]

print("=== 1) OBJ geometries ===")
instances_dir = result.get("instances_dir")
if instances_dir is not None and Path(instances_dir).exists():
    objs = sorted(Path(instances_dir).glob("*.obj"))
    print("instances_dir:", instances_dir)
    print("instance OBJ count:", len(objs))
    print("example instance OBJs:")
    for p in objs[:5]:
        print("  ", p)
else:
    for key in ("scene_obj", "beams_obj"):
        p = result.get(key)
        if p:
            print(f"{key}:", p)

print("\n=== 2) hitObjectId -> beam mapping (+ properties) ===")
mapping = list(getattr(sidecar, "beam_instance_hit_mapping", []) or [])
if mapping:
    print("beam_instance_hit_mapping rows:", len(mapping))
    print("example rows:")
    for row in mapping[:5]:
        print(
            "  part_index={helios_part_index}, instance_id={instance_id}, global_id={global_id}, "
            "profile={profile_name}, material={material_name}, name={name}".format(
                helios_part_index=row.get("helios_part_index"),
                instance_id=row.get("instance_id"),
                global_id=row.get("global_id"),
                profile_name=row.get("profile_name"),
                material_name=row.get("material_name"),
                name=row.get("name"),
            )
        )
else:
    beams = list(getattr(sidecar, "beams", []) or [])
    print("No explicit beam_instance_hit_mapping in this sidecar.")
    print("beam rows available:", len(beams))
    print("example beam rows:")
    for row in beams[:5]:
        print(
            "  instance_id={instance_id}, global_id={global_id}, profile={profile_name}, material={material_name}, name={name}".format(
                instance_id=row.get("instance_id"),
                global_id=row.get("global_id"),
                profile_name=row.get("profile_name"),
                material_name=row.get("material_name"),
                name=row.get("name"),
            )
        )

print("\n=== 3) point cloud outputs ===")
las_files = sorted((run_dir / "sim_output").rglob("*.las")) if (run_dir / "sim_output").exists() else []
if las_files:
    print("LAS file count:", len(las_files))
    print("example LAS files:")
    for p in las_files[:5]:
        print("  ", p)
else:
    print("No LAS files found in this run. Re-run with RUN_HELIOS=True and HELIOS++ available.")

pc2beam_txt = result.get("pc2beam_input_txt")
if pc2beam_txt is not None and Path(pc2beam_txt).exists():
    print("pc2beam_input_txt:", pc2beam_txt)
    with open(pc2beam_txt, "r", encoding="utf-8") as fh:
        print("first 3 rows:")
        for _ in range(3):
            line = fh.readline().strip()
            if not line:
                break
            print("  ", line)
else:
    print("No pc2beam_input_txt generated in this run (requires RUN_HELIOS=True).")

print("\n=== 4) ground truth ===")
gt_yaml = result.get("ground_truth_yaml")
if gt_yaml is not None and Path(gt_yaml).exists():
    print("ground_truth_yaml:", gt_yaml)
else:
    print("Ground-truth YAML path missing or file not found.")

=== 1) OBJ geometries ===
instances_dir: /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances
instance OBJ count: 91
example instance OBJs:
   /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances/i00001.obj
   /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances/i00002.obj
   /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances/i00003.obj
   /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances/i00004.obj
   /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances/i00005.obj

=== 2) hitObjectId -> beam mapping (+ properties) ===
beam_instance_hit_mapping rows: 80
example rows:
  part_index=11, instance_id=12, global_id=0HNx9aGbXDrOgS2t_z_MNs, profile=HE 260 A, material=S235 | EN 10025-2:2004-11, name=Member 1
  part_index=12, instance_id=13, global_

## intermediate visualization — mesh + scanner layout

In [6]:
scan_cfg = OmegaConf.load(str(scanners_yaml))
positions = [tuple(map(float, s.position)) for s in scan_cfg.scanners]
orientations = []
labels = []
for s in scan_cfg.scanners:
    att = s.get("attitude_deg") or {}
    orientations.append(
        (float(att.get("yaw", 0)), float(att.get("pitch", 0)), float(att.get("roll", 0)))
    )
    labels.append(str(s.get("id", "scanner")))

mesh_path = result.get("beams_obj")
if mesh_path is not None:
    v, f, _ = load_mesh_for_preview(mesh_path, max_triangles=40_000)
    fig_scene = plot_scanners_and_mesh(
        v,
        f,
        positions,
        scanner_orientations_deg=orientations,
        scanner_labels=labels,
        vector_length=4.0,
        mesh_title="Tessellated IFC geometry + dummy scanner stations",
        max_triangles=35_000,
    )
    fig_scene.show(renderer="notebook_connected")
else:
    print(
        "Merged mesh preview skipped (per-instance OBJ mode). "
        "Instance meshes live under:", result.get("instances_dir")
    )

Merged mesh preview skipped (per-instance OBJ mode). Instance meshes live under: /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/data/sceneparts/pc2beam/instances


## simulated scan — LAS point cloud (when `RUN_HELIOS` was True)

In [7]:
las_path = find_first_las(result["sim_output_dir"])
if las_path is None:
    print("No LAS found. Enable RUN_HELIOS after installing HELIOS++, or inspect sim_output_dir manually:", result["sim_output_dir"])
else:
    print("LAS:", las_path)
    print("LAS dimensions:", las_dimension_names(las_path))
    scan = read_las_scan(las_path)
    pts = scan.points
    print("points:", pts.shape[0])

    color_by = "uniform"
    features = None
    instances = None

    if scan.hit_object_id is not None:
        hid = np.asarray(scan.hit_object_id, dtype=np.int64)
        nu = len(np.unique(hid))
        print("hitObjectId unique values:", nu)
        if nu > 1:
            mapping_rows = list(getattr(result["sidecar"], "beam_instance_hit_mapping", []) or [])
            part_to_instance = {}
            for row in mapping_rows:
                part_idx = row.get("helios_part_index")
                inst_id = row.get("instance_id")
                if part_idx is None or inst_id is None:
                    continue
                part_to_instance[int(part_idx)] = int(inst_id)

            if part_to_instance:
                mapped = np.full(hid.shape, -1, dtype=np.int32)
                for part_idx, inst_id in part_to_instance.items():
                    mapped[hid == int(part_idx)] = int(inst_id)
                if np.any(mapped >= 0):
                    instances = mapped
                    color_by = "instance"
                    print("sample hitObjectId -> instance_id mapping:")
                    for h in sorted(np.unique(hid))[:12]:
                        print(f"  part_index {h} -> instance_id {part_to_instance.get(int(h), None)}")
                else:
                    print("No hitObjectId values matched beam_instance_hit_mapping; using fallback coloring.")
            else:
                print(
                    "No beam_instance_hit_mapping in sidecar; cannot enforce GT-instance alignment from hitObjectId. "
                    "Using fallback coloring."
                )
        else:
            print(
                "hitObjectId is constant (often 0 with a single merged OBJ). "
                "Re-run pipeline with helios_one_obj_per_instance=True (default here)."
            )

    if instances is None and scan.intensity is not None:
        color_by = "s1"
        s1 = scan.intensity.astype(np.float32)
        s1 = (s1 - s1.min()) / (np.ptp(s1) + 1e-9)
        features = {"s1": s1}

    fig_scan = plot_point_cloud(
        pts,
        instances=instances,
        features=features,
        mode="points",
        color_by=color_by,
        show_vectors=False,
        title="Simulated TLS (color = IFC instance_id)" if instances is not None else "Simulated TLS point cloud",
        max_points=50_000,
        ortho_view=True,
    )

    gt_yaml = result.get("ground_truth_yaml")
    if gt_yaml is None or not Path(gt_yaml).exists():
        print("No ground-truth YAML found for GT-instance diagnostics/overlay.")
    else:
        try:
            gt_data = OmegaConf.to_container(OmegaConf.load(gt_yaml), resolve=True)
            gt_keys_sorted = sorted(gt_data.keys(), key=lambda k: int(k))
            gt_instance_ids = {int(k) for k in gt_keys_sorted}

            point_instance_ids = set()
            if instances is not None:
                point_instance_ids = {int(v) for v in np.unique(instances) if int(v) >= 0}

            print("GT instance ids (sample):", sorted(gt_instance_ids)[:12])
            print("Point instance ids (sample):", sorted(point_instance_ids)[:12])
            print("instances_without_gt:", sorted(point_instance_ids - gt_instance_ids)[:20])
            print("gt_without_points:", sorted(gt_instance_ids - point_instance_ids)[:20])

            palette = [
                "rgb(31,119,180)", "rgb(255,127,14)", "rgb(44,160,44)", "rgb(214,39,40)",
                "rgb(148,103,189)", "rgb(140,86,75)", "rgb(227,119,194)", "rgb(127,127,127)",
                "rgb(188,189,34)", "rgb(23,190,207)",
            ]
            label_space = sorted(point_instance_ids | gt_instance_ids)
            color_map = {lbl: palette[i % len(palette)] for i, lbl in enumerate(label_space)}

            for gt_key in gt_keys_sorted:
                rec = gt_data[gt_key]
                inst_id = int(gt_key)
                sx, sy, sz = [float(v) for v in rec["start"]]
                ex, ey, ez = [float(v) for v in rec["end"]]
                beam_type = rec.get("beam_type", "unknown")
                color = color_map.get(inst_id, "rgb(255,0,255)")
                lg = f"gt_beam_{inst_id}"

                fig_scan.add_trace(
                    go.Scatter3d(
                        x=[sx, ex],
                        y=[sy, ey],
                        z=[sz, ez],
                        mode="lines",
                        line=dict(color=color, width=6),
                        name=f"GT beam {inst_id}",
                        legendgroup=lg,
                        showlegend=True,
                        hoverinfo="text",
                        text=[
                            f"beam {inst_id}<br>type={beam_type}<br>start=({sx:.3f}, {sy:.3f}, {sz:.3f})",
                            f"beam {inst_id}<br>type={beam_type}<br>end=({ex:.3f}, {ey:.3f}, {ez:.3f})",
                        ],
                    )
                )
                fig_scan.add_trace(
                    go.Scatter3d(
                        x=[sx, ex],
                        y=[sy, ey],
                        z=[sz, ez],
                        mode="markers",
                        marker=dict(
                            size=5,
                            color=color,
                            opacity=1.0,
                            symbol="diamond",
                            line=dict(color="rgb(0,0,0)", width=1),
                        ),
                        name=f"GT anchors {inst_id}",
                        legendgroup=lg,
                        showlegend=False,
                        hoverinfo="text",
                        text=[
                            f"beam {inst_id} start<br>({sx:.3f}, {sy:.3f}, {sz:.3f})",
                            f"beam {inst_id} end<br>({ex:.3f}, {ey:.3f}, {ez:.3f})",
                        ],
                    )
                )

            print(f"GT overlay added from {gt_yaml} (GT beams: {len(gt_keys_sorted)})")
        except Exception as e:
            print(f"Failed to load GT diagnostics/overlay from {gt_yaml}: {e}")

    fig_scan.show(renderer="notebook_connected")

LAS: /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/sim_output/pc2beam_ifc_tls/2026-04-16_10-46-11/leg000_points.las
LAS dimensions: ['X', 'Y', 'Z', 'intensity', 'return_number', 'number_of_returns', 'synthetic', 'key_point', 'withheld', 'overlap', 'scanner_channel', 'scan_direction_flag', 'edge_of_flight_line', 'classification', 'user_data', 'scan_angle', 'point_source_id', 'gps_time', 'echo_width', 'fullwaveIndex', 'hitObjectId', 'heliosAmplitude', 'ExtraBytes']
points: 243793
hitObjectId unique values: 90
sample hitObjectId -> instance_id mapping:
  part_index 0 -> instance_id None
  part_index 1 -> instance_id None
  part_index 2 -> instance_id None
  part_index 3 -> instance_id None
  part_index 4 -> instance_id None
  part_index 5 -> instance_id None
  part_index 6 -> instance_id None
  part_index 7 -> instance_id None
  part_index 8 -> instance_id None
  part_index 9 -> instance_id None
  part_index 10 -> instance_id None
  part_index 11 -> instance_id 12
GT instanc

In [8]:
# GT overlay per beam instance (separate line + anchors per beam, shared color per beam)
# Run this AFTER the LAS visualization cell so `fig_scan` and `result` already exist.

from pathlib import Path
from omegaconf import OmegaConf
import numpy as np
import plotly.graph_objects as go

if "fig_scan" not in globals() or "result" not in globals():
    print("Run the LAS visualization cell first (fig_scan/result not available).")
else:
    gt_yaml = result.get("ground_truth_yaml")
    if gt_yaml is None or not Path(gt_yaml).exists():
        print("No ground-truth YAML found for per-beam overlay.")
    else:
        try:
            gt_data = OmegaConf.to_container(OmegaConf.load(gt_yaml), resolve=True)
            gt_keys_sorted = sorted(gt_data.keys(), key=lambda k: int(k))

            # Build color mapping from current instance label space when available
            if "instances" in globals() and instances is not None:
                point_labels = sorted(int(v) for v in np.unique(np.asarray(instances, dtype=np.int64)))
            else:
                point_labels = [int(k) for k in gt_keys_sorted]

            palette = [
                "rgb(31,119,180)", "rgb(255,127,14)", "rgb(44,160,44)", "rgb(214,39,40)",
                "rgb(148,103,189)", "rgb(140,86,75)", "rgb(227,119,194)", "rgb(127,127,127)",
                "rgb(188,189,34)", "rgb(23,190,207)"
            ]
            color_map = {lbl: palette[i % len(palette)] for i, lbl in enumerate(point_labels)}

            # Optional sidecar mapping: part_index -> instance_id, and inverse instance_id -> part_index
            p2i = {}
            i2p = {}
            sidecar_map = list(getattr(result["sidecar"], "beam_instance_hit_mapping", []) or [])
            for row in sidecar_map:
                part_idx = row.get("helios_part_index")
                inst_id = row.get("instance_id")
                if part_idx is None or inst_id is None:
                    continue
                part_idx = int(part_idx)
                inst_id = int(inst_id)
                p2i[part_idx] = inst_id
                if inst_id not in i2p:
                    i2p[inst_id] = part_idx

            added = 0
            for gt_key in gt_keys_sorted:
                rec = gt_data[gt_key]
                sx, sy, sz = [float(v) for v in rec["start"]]
                ex, ey, ez = [float(v) for v in rec["end"]]
                beam_type = rec.get("beam_type", "unknown")
                inst_id = int(gt_key)

                # If points are colored by part_index, map instance_id -> part_index for color lookup
                label_for_color = i2p.get(inst_id, inst_id)
                color = color_map.get(label_for_color, palette[inst_id % len(palette)])
                legend_group = f"gt_beam_{inst_id}"

                # One line trace per beam
                fig_scan.add_trace(
                    go.Scatter3d(
                        x=[sx, ex],
                        y=[sy, ey],
                        z=[sz, ez],
                        mode="lines",
                        line=dict(color=color, width=6),
                        name=f"GT beam {inst_id}",
                        legendgroup=legend_group,
                        showlegend=True,
                        hoverinfo="text",
                        text=[
                            f"beam {inst_id}<br>type={beam_type}<br>start=({sx:.3f}, {sy:.3f}, {sz:.3f})",
                            f"beam {inst_id}<br>type={beam_type}<br>end=({ex:.3f}, {ey:.3f}, {ez:.3f})",
                        ],
                    )
                )

                # One anchor trace per beam (start + end)
                fig_scan.add_trace(
                    go.Scatter3d(
                        x=[sx, ex],
                        y=[sy, ey],
                        z=[sz, ez],
                        mode="markers",
                        marker=dict(
                            size=5,
                            color=color,
                            opacity=1.0,
                            symbol="diamond",
                            line=dict(color="rgb(0,0,0)", width=1),
                        ),
                        name=f"GT anchors {inst_id}",
                        legendgroup=legend_group,
                        showlegend=False,  # keep one legend item per beam
                        hoverinfo="text",
                        text=[
                            f"beam {inst_id} start<br>({sx:.3f}, {sy:.3f}, {sz:.3f})",
                            f"beam {inst_id} end<br>({ex:.3f}, {ey:.3f}, {ez:.3f})",
                        ],
                    )
                )
                added += 1

            print(f"Added per-beam GT overlay from {gt_yaml} (beams: {added})")
            fig_scan.show(renderer="notebook_connected")

        except Exception as e:
            print(f"Failed to load GT overlay from {gt_yaml}: {e}")

Added per-beam GT overlay from /Users/fnoi/Code/pc2beam/output/helios_pc2beam/cade0181c15a/pc2beam_input/model_0_z_up_gt.yaml (beams: 80)


In [ ]:
# Phase 1 validation: robust s2 + quantitative evaluation
from pc2beam.data import PointCloud
from pc2beam.evaluation import evaluate_s2_against_ground_truth, summarize_s2_metrics

pc2beam_input_txt = result.get("pc2beam_input_txt")
gt_yaml = result.get("ground_truth_yaml")

if not pc2beam_input_txt or not gt_yaml:
    raise ValueError("Run the HELIOS pipeline cell first so pc2beam_input_txt and ground_truth_yaml are available.")

pc = PointCloud.from_txt(pc2beam_input_txt)
pc.compute_s2(
    distance_threshold=0.01,
    ransac_n=3,
    num_iterations=1000,
    min_points_per_instance=20,
    min_plane_inliers=10,
    angle_min_deg=30.0,
    angle_max_deg=150.0,
    enable_fallback=True,
)

metrics_df = evaluate_s2_against_ground_truth(
    points=pc.points,
    instances=pc.instances,
    s2_features=pc.features["s2"],
    ground_truth_yaml=gt_yaml,
)
summary = summarize_s2_metrics(metrics_df)
print(summary)

# Inspect worst angular outliers among valid predictions
if not metrics_df.empty:
    display(
        metrics_df.sort_values("axis_angle_error_deg", ascending=False, na_position="last").head(10)
    )